# Training Helmcheck K3 di Google Colab (dataset.zip dari Google Drive)

Notebook ini melatih YOLOv8 dari `dataset.zip` yang sudah ada di Google Drive, lalu menyiapkan hasilnya untuk website.

**Sebelum mulai:**
1. Menu `Runtime → Change runtime type → T4 GPU → Save`.
2. Pastikan `dataset.zip` ada di Google Drive. Kalau letaknya bukan di folder utama *My Drive*, ubah `ZIP_PATH` di Langkah 3.
3. Jalankan sel satu per satu dari atas (`Shift + Enter`).

Isi zip yang diharapkan (format YOLO, sudah dibagi):

```
dataset/
├── data.yaml
├── images/{train,val,test}
└── labels/{train,val,test}     kelas: 0 helmet, 1 head, 2 person
```

## Langkah 1 — Cek GPU
Harus muncul nama GPU, misalnya *Tesla T4*. Kalau tertulis `False`, ulangi pengaturan runtime di atas.

In [ ]:
import torch
print("GPU tersedia:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Langkah 2 — Pasang Ultralytics

In [ ]:
%pip install -q ultralytics

## Langkah 3 — Hubungkan Google Drive dan atur konfigurasi

Saat diminta, pilih akun Google yang menyimpan `dataset.zip` lalu izinkan akses.

Hasil training disimpan **langsung di Drive** (`OUT_DIR`). Jadi kalau Colab terputus, bobot terakhir tidak hilang dan training bisa dilanjutkan (lihat Langkah 7b).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")   # ubah kalau zip ada di folder lain
OUT_DIR = Path("/content/drive/MyDrive/helmcheck")       # hasil training di Drive
DATA = Path("/content/dataset")                          # dataset diekstrak ke disk lokal Colab (lebih cepat)

EPOCHS = 50          # T4: sekitar 1 menit per epoch untuk 4000 gambar
IMGSZ = 640
BATCH = 16           # turunkan ke 8 kalau muncul "CUDA out of memory"
BASE_MODEL = "yolov8n.pt"
NAMES = ["helmet", "no_helmet"]   # nama kelas yang dipakai website

print("Zip ditemukan:", ZIP_PATH.exists(), f"({ZIP_PATH.stat().st_size / 1e9:.2f} GB)" if ZIP_PATH.exists() else "")

## Langkah 4 — Ekstrak dataset
Sekitar 1–2 menit untuk zip 1,3 GB.

In [ ]:
import shutil, zipfile

shutil.rmtree(DATA, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content")

for split in ["train", "val", "test"]:
    n_img = len(list((DATA / "images" / split).glob("*")))
    n_lbl = len(list((DATA / "labels" / split).glob("*.txt")))
    print(f"{split:5s}: {n_img} gambar, {n_lbl} label")
print((DATA / "data.yaml").read_text())

## Langkah 5 — Rapikan label untuk website

Tiga perbaikan:
1. Kelas `head` (kepala tanpa helm) diganti nama menjadi `no_helmet`, karena website menandai merah kelas dengan nama itu. Angkanya tetap `1`, jadi isi file label tidak perlu diubah.
2. Kelas `person` (2) dibuang. Jumlahnya sedikit dan tidak dipakai website.
3. `data.yaml` ditulis ulang dengan path Colab, dan file `.cache` lama dihapus karena berisi path Windows.

In [ ]:
from collections import Counter

before, after = Counter(), Counter()
for txt in (DATA / "labels").glob("*/*.txt"):
    lines = txt.read_text().splitlines()
    before.update(l.split()[0] for l in lines if l.strip())
    keep = [l for l in lines if l.strip() and l.split()[0] in {"0", "1"}]
    after.update(l.split()[0] for l in keep)
    txt.write_text("\n".join(keep))

for cache in (DATA / "labels").glob("*.cache"):
    cache.unlink()

DATA_YAML = DATA / "data.yaml"
DATA_YAML.write_text(
    f"path: {DATA}\ntrain: images/train\nval: images/val\ntest: images/test\nnames: {NAMES}\n"
)
print("Sebelum:", dict(sorted(before.items())), " (0=helmet, 1=head, 2=person)")
print("Sesudah:", dict(sorted(after.items())), " (0=helmet, 1=no_helmet)")
print(DATA_YAML.read_text())

## Langkah 6 — Cek satu gambar
Kotak hijau = helm, merah = tanpa helm. Kotak harus pas di kepala; kalau meleset, label rusak dan training tidak akan berhasil.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

img_path = random.Random(7).choice(sorted((DATA / "images" / "train").glob("*")))
img = Image.open(img_path).convert("RGB"); W, H = img.size; d = ImageDraw.Draw(img)
for line in (DATA / "labels" / "train" / (img_path.stem + ".txt")).read_text().splitlines():
    c, xc, yc, w, h = map(float, line.split())
    d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline=["lime", "red"][int(c)], width=2)
plt.figure(figsize=(6, 6)); plt.imshow(img); plt.axis("off"); plt.title(img_path.name); plt.show()

## Langkah 7 — Training

Biarkan tab Colab tetap terbuka selama training. Colab gratis memutus sesi yang ditinggal terlalu lama.
Progres per epoch tampil di bawah sel: `box_loss`, `cls_loss`, dan `dfl_loss` harus turun, sedangkan `mAP50` harus naik.

`patience=15` menghentikan training lebih awal kalau mAP validasi tidak membaik selama 15 epoch.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=15,
    seed=42,
    project=str(OUT_DIR / "runs"),   # path absolut di Drive -> OUT_DIR/runs/helmet
    name="helmet",
    exist_ok=True,
)
RUN = Path(model.trainer.save_dir)
print("Hasil training:", RUN)

### Langkah 7b — Kalau Colab terputus di tengah training

Jalankan ulang Langkah 1–6 (dataset perlu diekstrak lagi), **lewati** Langkah 7, lalu jalankan sel di bawah ini. Training akan dilanjutkan dari epoch terakhir yang tersimpan di Drive.

In [ ]:
# Hapus tanda # di dua baris berikut hanya kalau training terputus:
# model = YOLO(OUT_DIR / "runs" / "helmet" / "weights" / "last.pt"); model.train(resume=True)
# RUN = Path(model.trainer.save_dir)

## Langkah 8 — Lihat kurva training
Loss train dan val yang sama-sama turun berarti model belajar dengan sehat. Kalau val loss naik sementara train loss turun, model mulai menghafal (overfitting).

In [ ]:
from IPython.display import Image as Show, display
RUN = OUT_DIR / "runs" / "helmet"
display(Show(filename=str(RUN / "results.png"), width=1000))

## Langkah 9 — Evaluasi dan simpan `metrics.json` untuk website

- **Validasi:** angka utama yang tampil di website.
- **Test:** 500 gambar yang belum pernah dipakai sama sekali, jadi gambaran paling jujur untuk performa di lapangan.
- **Confusion matrix:** dihitung pada ambang keyakinan 0,4, sama dengan default website.

In [ ]:
import json
import numpy as np
import pandas as pd

best = YOLO(RUN / "weights" / "best.pt")

def summarize(m):
    per_class = [{"name": NAMES[int(c)], "precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
                  "map50": float(m.box.ap50[i]), "map50_95": float(m.box.ap[i])} for i, c in enumerate(m.box.ap_class_index)]
    overall = {"precision": float(m.box.mp), "recall": float(m.box.mr), "map50": float(m.box.map50), "map50_95": float(m.box.map)}
    return overall, per_class

m_val = best.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
m_test = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
overall, per_class = summarize(m_val)
test_overall, _ = summarize(m_test)

print("VALIDASI"); display(pd.DataFrame(per_class).set_index("name").round(3))
print("TEST    ", {k: round(v, 3) for k, v in test_overall.items()})

In [ ]:
CONF_CM = 0.4
# plots=True wajib: Ultralytics hanya mengisi confusion matrix kalau plot dibuat
cmv = best.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, conf=CONF_CM, plots=True,
               verbose=False, project=str(RUN), name="val", exist_ok=True)
cm = cmv.confusion_matrix.matrix

BINS = 20
heat = np.zeros((BINS, BINS))
for lbl in (DATA / "labels" / "train").glob("*.txt"):
    for line in lbl.read_text().splitlines():
        _, xc, yc, *_ = map(float, line.split())
        heat[min(int(yc * BINS), BINS - 1), min(int(xc * BINS), BINS - 1)] += 1

metrics = {
    "names": NAMES,
    "overall": overall,
    "per_class": per_class,
    "test_overall": test_overall,
    "confusion": {"labels": NAMES + ["background"], "matrix": cm.astype(int).tolist()},
    "label_heatmap": heat.astype(int).tolist(),
    "counts": {s: len(list((DATA / "images" / s).glob("*"))) for s in ["train", "val"]},
    "config": {"base_model": BASE_MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "conf_cm": CONF_CM},
}
(RUN / "metrics.json").write_text(json.dumps(metrics, indent=1))
print("Tersimpan:", RUN / "metrics.json")

## Langkah 10 — Coba deteksi pada gambar test

In [ ]:
samples = random.Random(3).sample(sorted((DATA / "images" / "test").glob("*")), 4)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, p in zip(axes, samples):
    r = best.predict(p, conf=0.4, verbose=False)[0]
    n_no = int((r.boxes.cls == 1).sum())
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f"{n_no} orang tanpa helm" if n_no else "semua pakai helm", color="red" if n_no else "green")
    ax.axis("off")
plt.show()

## Langkah 11 — Bungkus hasil untuk website

Sel ini membuat `helmet_run.zip` di folder `helmcheck` di Drive, lalu mencoba mengunduhnya. Kalau unduhan tidak muncul, ambil file itu dari Google Drive.

**Memasang di website (di laptop):**
1. Ekstrak `helmet_run.zip` ke folder project `yolov8-helm-detect`, sehingga ada `runs/helmet/weights/best.pt`.
2. Jalankan `python app.py`, lalu buka http://127.0.0.1:5000.

In [ ]:
zip_path = shutil.make_archive(str(OUT_DIR / "helmet_run"), "zip", root_dir=RUN.parent, base_dir=RUN.name)
print("Zip:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
from google.colab import files
files.download(zip_path)